# IPL Analytics — EDA & Processing Pipeline
**Data:** Ball-by-ball IPL data 2008–2025 (`data/raw/IPL.csv`)  
**Output:** Processed CSVs + trained win probability model

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='darkgrid', palette='viridis')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.facecolor'] = '#0d0d20'
plt.rcParams['figure.facecolor'] = '#080810'
plt.rcParams['text.color'] = '#e8e8ff'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'

BASE_DIR = Path('..').resolve()
RAW_PATH = BASE_DIR / 'data' / 'raw' / 'IPL.csv'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('Paths OK:', RAW_PATH.exists())

## 1. Load & Clean

In [ ]:
def clean_season(x):
    x = str(x)
    if '/' in x:
        last_part = x.split('/')[-1]
        return int('20' + last_part) if len(last_part) == 2 else int(last_part)
    return int(x)

df = pd.read_csv(RAW_PATH, low_memory=False)

print('Raw shape:', df.shape)
print('Columns:', df.columns.tolist())

In [ ]:
# Season cleaning
df['season'] = df['season'].apply(clean_season)
df['season'] = pd.to_numeric(df['season'], errors='coerce')
df = df.dropna(subset=['season'])
df['season'] = df['season'].astype(int)

# Team name normalization
team_map = {
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Deccan Chargers': 'Sunrisers Hyderabad',
    'Rising Pune Supergiant': 'Rising Pune Supergiants',
    'Kochi Tuskers Kerala': 'Other',
    'Pune Warriors': 'Other',
    'Gujarat Lions': 'Other'
}

venue_map = {
    'M.Chinnaswamy Stadium': 'M Chinnaswamy Stadium',
    'Feroz Shah Kotla': 'Arun Jaitley Stadium',
    'Sardar Patel Stadium': 'Narendra Modi Stadium',
    'Punjab Cricket Association Stadium': 'PCA IS Bindra Stadium',
    'Punjab Cricket Association IS Bindra Stadium': 'PCA IS Bindra Stadium',
}

str_cols = df.select_dtypes(include='object').columns
df[str_cols] = df[str_cols].fillna('Missing')

for col in ['batting_team', 'bowling_team', 'match_won_by']:
    df[col] = df[col].replace(team_map)

df['venue'] = df['venue'].str.split(',').str[0].replace(venue_map)
df['date']  = pd.to_datetime(df['date'])

# Derived columns
df['current_score']    = df.groupby(['match_id','innings'])['runs_total'].cumsum()
df['current_run_rate'] = (df['current_score'] / df['ball_no']) * 6
df['is_wicket']        = df['player_out'].apply(lambda x: 0 if x == 'Missing' else 1)
df['wickets_fallen']   = df.groupby(['match_id','innings'])['is_wicket'].cumsum()
df['is_four']          = (df['runs_batter'] == 4).astype(int)
df['is_six']           = (df['runs_batter'] == 6).astype(int)

print('Clean shape:', df.shape)
df.head(3)

## 2. EDA — Data Quality

In [ ]:
print('Seasons:', sorted(df['season'].unique()))
print('Teams:', sorted(df['batting_team'].unique()))
print('Total deliveries:', len(df))
print('Total matches:', df['match_id'].nunique())
print('\nNull counts:')
print(df.isnull().sum()[df.isnull().sum() > 0])

## 3. EDA — Season Trends

In [ ]:
season_runs = df.groupby('season')['runs_total'].sum().reset_index()
season_sixes = df.groupby('season')['is_six'].sum().reset_index()
season_matches = df.groupby('season')['match_id'].nunique().reset_index()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].bar(season_runs['season'], season_runs['runs_total'], color='#7c3aed')
axes[0].set_title('Total Runs Per Season', color='#e8e8ff')
axes[0].set_xlabel('Season')

axes[1].bar(season_sixes['season'], season_sixes['is_six'], color='#f59e0b')
axes[1].set_title('Sixes Per Season', color='#e8e8ff')

axes[2].bar(season_matches['season'], season_matches['match_id'], color='#3b82f6')
axes[2].set_title('Matches Per Season', color='#e8e8ff')

plt.tight_layout()
plt.show()

## 4. EDA — Batting Patterns

In [ ]:
# Top run scorers all time
top_batters = df.groupby('batter')['runs_batter'].sum().sort_values(ascending=False).head(15)

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

axes[0].barh(top_batters.index[::-1], top_batters.values[::-1], color='#7c3aed')
axes[0].set_title('Top 15 Run Scorers (All Time)', color='#e8e8ff')
axes[0].set_xlabel('Runs')

# Run distribution per over
over_runs = df.groupby('over')['runs_total'].mean().reset_index()
axes[1].plot(over_runs['over'], over_runs['runs_total'], color='#f59e0b', linewidth=2.5)
axes[1].fill_between(over_runs['over'], over_runs['runs_total'], alpha=0.3, color='#f59e0b')
axes[1].set_title('Avg Runs Per Over (Powerplay vs Death)', color='#e8e8ff')
axes[1].set_xlabel('Over')

plt.tight_layout()
plt.show()

## 5. EDA — Bowling Patterns

In [ ]:
# Top wicket takers
top_bowlers = df.groupby('bowler')['bowler_wicket'].sum().sort_values(ascending=False).head(15)

# Wickets by over
wicket_by_over = df.groupby('over')['is_wicket'].sum().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

axes[0].barh(top_bowlers.index[::-1], top_bowlers.values[::-1], color='#ef4444')
axes[0].set_title('Top 15 Wicket Takers (All Time)', color='#e8e8ff')
axes[0].set_xlabel('Wickets')

axes[1].bar(wicket_by_over['over'], wicket_by_over['is_wicket'], color='#ef4444')
axes[1].set_title('Wickets Distribution By Over', color='#e8e8ff')
axes[1].set_xlabel('Over')

plt.tight_layout()
plt.show()

## 6. EDA — Team & Toss Analysis

In [ ]:
matches = df[['match_id','batting_team','bowling_team','match_won_by',
              'toss_winner','toss_decision','venue','season']].drop_duplicates('match_id')

# Win % by team
all_teams = pd.concat([
    matches[['match_id','batting_team','match_won_by']].rename(columns={'batting_team':'team'}),
    matches[['match_id','bowling_team','match_won_by']].rename(columns={'bowling_team':'team'})
])
all_teams = all_teams[all_teams['team'] != 'Other']

total = all_teams.groupby('team').size()
wins  = all_teams[all_teams['team'] == all_teams['match_won_by']].groupby('team').size()
win_pct = (wins / total * 100).sort_values(ascending=False).dropna()

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

axes[0].barh(win_pct.index[::-1], win_pct.values[::-1], color='#3b82f6')
axes[0].set_title('Win % By Team (All Time)', color='#e8e8ff')
axes[0].set_xlabel('Win %')

# Toss decision distribution
toss_counts = matches['toss_decision'].value_counts()
axes[1].pie(toss_counts.values, labels=toss_counts.index,
            colors=['#7c3aed','#f59e0b'], autopct='%1.1f%%',
            textprops={'color':'#e8e8ff'})
axes[1].set_title('Toss Decision Split', color='#e8e8ff')

plt.tight_layout()
plt.show()

## 7. EDA — Venue Analysis

In [ ]:
venue_matches = matches.groupby('venue')['match_id'].count().sort_values(ascending=False).head(12)

# First innings scores by venue
inn1 = df[df['innings'] == 1].groupby(['match_id','venue'])['runs_total'].sum().reset_index()
avg_inn1 = inn1.groupby('venue')['runs_total'].mean().sort_values(ascending=False).head(12)

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

axes[0].barh(venue_matches.index[::-1], venue_matches.values[::-1], color='#10b981')
axes[0].set_title('Most Matches Hosted', color='#e8e8ff')

axes[1].barh(avg_inn1.index[::-1], avg_inn1.values[::-1], color='#f59e0b')
axes[1].set_title('Avg 1st Innings Score By Venue', color='#e8e8ff')

plt.tight_layout()
plt.show()

## 8. Win Probability — Feature Engineering

In [ ]:
# Get 2nd innings only (chasing team — we predict if they win)
inn2 = df[df['innings'] == 2].copy()

# Get target for each match (1st innings total + 1)
inn1_totals = df[df['innings'] == 1].groupby('match_id')['runs_total'].sum().reset_index()
inn1_totals.columns = ['match_id', 'target']
inn1_totals['target'] = inn1_totals['target'] + 1

inn2 = inn2.merge(inn1_totals, on='match_id', how='left')

# Features per ball
inn2['runs_needed']     = inn2['target'] - inn2['current_score']
inn2['balls_remaining'] = 120 - (inn2['over'] * 6 + (inn2['ball_no'] % 6))
inn2['balls_remaining'] = inn2['balls_remaining'].clip(lower=1)
inn2['rrr']             = (inn2['runs_needed'] / inn2['balls_remaining']) * 6
inn2['crr']             = inn2['current_run_rate']
inn2['wickets_left']    = 10 - inn2['wickets_fallen']
inn2['run_rate_diff']   = inn2['crr'] - inn2['rrr']
inn2['wicket_pressure'] = inn2['wickets_fallen'] / 10
inn2['over_progress']   = inn2['over'] / 20

# Label: did batting team win?
inn2['batting_won'] = (inn2['batting_team'] == inn2['match_won_by']).astype(int)

# Drop rows where target is null
inn2 = inn2.dropna(subset=['target','runs_needed','rrr'])

print('Inn2 shape:', inn2.shape)
print('Win rate:', inn2['batting_won'].mean().round(3))

In [ ]:
# Feature correlation with outcome
features = ['current_score','wickets_fallen','wickets_left','runs_needed',
            'balls_remaining','rrr','crr','run_rate_diff',
            'wicket_pressure','over_progress']

corr = inn2[features + ['batting_won']].corr()['batting_won'].drop('batting_won').sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#ef4444' if v < 0 else '#10b981' for v in corr.values]
ax.barh(corr.index, corr.values, color=colors)
ax.set_title('Feature Correlation With Win Outcome', color='#e8e8ff')
ax.axvline(0, color='white', linewidth=0.8)
plt.tight_layout()
plt.show()

## 9. Win Probability — Model Training

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import joblib

FEATURES = [
    'current_score', 'wickets_fallen', 'wickets_left',
    'runs_needed', 'balls_remaining', 'rrr', 'crr',
    'run_rate_diff', 'wicket_pressure', 'over_progress'
]

X = inn2[FEATURES].replace([np.inf, -np.inf], np.nan).dropna()
y = inn2.loc[X.index, 'batting_won']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        random_state=42
    ))
])

pipeline.fit(X_train, y_train)

y_pred  = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print(f'Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_proba):.4f}')
print()
print(classification_report(y_test, y_pred))

In [ ]:
# Feature importance
model = pipeline.named_steps['model']
importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(importances.index, importances.values, color='#7c3aed')
ax.set_title('Feature Importances — Win Probability Model', color='#e8e8ff')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Calibration check — does 60% predicted actually win 60% of the time?
cal_df = pd.DataFrame({'prob': y_proba, 'actual': y_test.values})
cal_df['bucket'] = pd.cut(cal_df['prob'], bins=10)
cal = cal_df.groupby('bucket')['actual'].mean()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot([0,1],[0,1], '--', color='#94a3b8', label='Perfect calibration')
midpoints = [iv.mid for iv in cal.index]
ax.plot(midpoints, cal.values, 'o-', color='#f59e0b', linewidth=2, label='Model')
ax.set_title('Model Calibration Curve', color='#e8e8ff')
ax.set_xlabel('Predicted Probability')
ax.set_ylabel('Actual Win Rate')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Save model + feature list together
model_payload = {
    'pipeline': pipeline,
    'features': FEATURES
}

model_path = PROCESSED_DIR / 'win_probability_model.pkl'
joblib.dump(model_payload, model_path)
print('Model saved to:', model_path)

## 10. Save All Processed CSVs

In [ ]:
# Re-import your processing functions from src
import sys
sys.path.insert(0, str(BASE_DIR / 'data' / 'src'))
from data_processing import (
    load_and_clean,
    create_df_stats_batter,
    create_df_stats_bowler,
    create_df_stats_team,
    create_df_stats_venue,
    create_df_stats_season,
    create_df_stats_h2h_matches,
    create_df_stats_h2h_summary,
    create_df_stats_h2h_player_stats,
    create_df_player_vs_player,
    create_df_player_vs_team,
)

df = load_and_clean(RAW_PATH)

files = {
    'player_batting.csv':    create_df_stats_batter(df),
    'player_bowler.csv':     create_df_stats_bowler(df),
    'team_stats.csv':        create_df_stats_team(df),
    'venue_stats.csv':       create_df_stats_venue(df),
    'season_stats.csv':      create_df_stats_season(df),
    'h2h_matches.csv':       create_df_stats_h2h_matches(df),
    'h2hsummary.csv':        create_df_stats_h2h_summary(create_df_stats_h2h_matches(df)),
    'h2hplayer_stats.csv':   create_df_stats_h2h_player_stats(df),
    'player_vs_player.csv':  create_df_player_vs_player(df),
    'player_vs_team.csv':    create_df_player_vs_team(df),
}

for fname, frame in files.items():
    frame.to_csv(PROCESSED_DIR / fname, index=False)
    print(f'Saved {fname}: {frame.shape}')

print('\nAll done.')